# Stamp a Text Record
This sample creates a collection if necessary, stamps a text record, and verifies the resulting content identifier (CID).

The vBase API calculates the CID and returns the blockchain receipt. This basic example sends the text to vBase and uses the default stamped-file storage; the S3 producer samples show how to submit only a locally calculated CID.

In [ ]:
%%capture

%pip install --quiet vbase-api
!wget --quiet --output-document=collab_utils.py https://raw.githubusercontent.com/validityBase/vbase-py-samples-collab/main/samples/collab_utils.py

from vbase_api import VBaseAPIClient

from collab_utils import (
    get_required_env,
    try_add_user_secrets_to_env,
    wait_for_stamp,
)

try_add_user_secrets_to_env(["VBASE_API_KEY"])

In [ ]:
COLLECTION_NAME = "Google Colab Text Sample"
COLLECTION_DESCRIPTION = "Text records created by the vBase Google Colab sample."
RECORD = "A verifiable text record"

In [ ]:
client = VBaseAPIClient(api_key=get_required_env("VBASE_API_KEY"))

In [ ]:
collection = next(
    (
        item
        for item in client.get_collections()
        if item.name.casefold() == COLLECTION_NAME.casefold()
    ),
    None,
)

if collection is None:
    collection = client.create_collection(
        name=COLLECTION_NAME,
        description=COLLECTION_DESCRIPTION,
    )

print(f"Collection: {collection.name} ({collection.cid})")

In [ ]:
stamp = client.create_stamp(
    data=RECORD,
    file_name="text-record.txt",
    collection_cid=collection.cid,
)
receipt = stamp.commitment_receipt

print(f"Stamped CID: {receipt.object_cid}")
print(f"Timestamp: {receipt.timestamp}")
print(f"Transaction: {receipt.transaction_hash}")

In [ ]:
verified_receipt = wait_for_stamp(
    client,
    receipt.object_cid,
    collection.cid,
    filter_by_user=True,
)
print(f"Verified at: {verified_receipt.timestamp}")
print("The text record was verified successfully.")
client.close()